## Modeling Expected Score Progression 

### Problem Statement
Each student has three grades — G1, G2, G3. If a student's progress follows a perfectly arithmetic sequence, their scores increase or decrease by a constant difference d each period. Reality rarely matches this. The question is:

"Did each student perform better, worse, or exactly as expected in G3 based on their G1 and G2 trajectory — and can we identify who deviated downward unexpectedly?"

### Goals

* Calculate the expected G3 for every student based on G1 and G2
* Calculate the actual deviation — expected G3 minus actual G3
* Classify each student as On Track, Overachiever, or Underachiever
* Find which performance group has the most unexpected underachievers
* Check if underachievers share common attributes like high absences or failures

### Success Metrics

* Every student has a calculated expected G3 and deviation score. 
* At least one performance group is disproportionately full of underachievers
* Underachievers differ from On Track students on at least one attribute

### Steps and Hints

Step 1 — Calculate the common difference d per student
For each student, d is simply the difference between G2 and G1. This is the rate of change per period.

Hint: This is a single pandas column operation. No loop needed.

Step 2 — Calculate expected G3 per student
If scores follow an arithmetic sequence, expected G3 = G2 + d.

Hint: This is also a single column operation using the d column you just created.

Step 3 — Calculate deviation per student
Deviation = actual G3 minus expected G3.

* Negative deviation → student did worse than expected
* Zero deviation → student performed exactly as expected
* Positive deviation → student did better than expected

Hint: Another single column operation.

Step 4 — Classify each student
Create a new column progression_label with three categories:

* Underachiever → deviation is negative
* On Track → deviation is zero
* Overachiever → deviation is positive

Hint: Use np.select() or pd.cut() — think about which fits better here and why.

Step 5 — Count and percentage per label
How many students fall into each category? What percentage?

Hint: You already did this exact operation yesterday with performance groups.

Step 6 — Cross-check with performance groups
Which performance group has the highest proportion of underachievers?

Hint: Use groupby on both performance_group and progression_label together.

Step 7 — Compare underachievers vs on track students
Do underachievers have more absences or failures than On Track students?

Hint: Use groupby on progression_label with the same approach as yesterday's cliff group comparison.

### Importing

In [31]:
import pandas as pd
import utils

### Student_ID

In [32]:
df_full_math = pd.read_csv("student-mat.csv", sep=";")
df_full_math

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,2,2,services,services,...,5,5,4,4,5,4,11,9,9,9
391,MS,M,17,U,LE3,T,3,1,services,services,...,2,4,5,3,4,2,3,14,16,16
392,MS,M,21,R,GT3,T,1,1,other,other,...,5,5,3,3,3,3,3,10,8,7
393,MS,M,18,R,LE3,T,3,2,services,other,...,4,4,1,3,4,5,0,11,12,10


In [33]:
df_full_math = utils.add_unique_student_id(df_full_math, 'student_id', 'std') 

In [34]:
df_full_math 

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,student_id
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,std001
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,std002
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,std003
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,std004
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,std005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,2,2,services,services,...,5,4,4,5,4,11,9,9,9,std391
391,MS,M,17,U,LE3,T,3,1,services,services,...,4,5,3,4,2,3,14,16,16,std392
392,MS,M,21,R,GT3,T,1,1,other,other,...,5,3,3,3,3,3,10,8,7,std393
393,MS,M,18,R,LE3,T,3,2,services,other,...,4,1,3,4,5,0,11,12,10,std394


In [35]:
df_full_math = utils.move_std_id(df_full_math, 'student_id') 

In [36]:
df_full_math.head() 

,student_id,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,std001,GP,F,18,U,GT3,A,4,4,at_home,...,4,3,4,1,1,3,6,5,6,6
1,std002,GP,F,17,U,GT3,T,1,1,at_home,...,5,3,3,1,1,3,4,5,5,6
2,std003,GP,F,15,U,LE3,T,1,1,at_home,...,4,3,2,2,3,3,10,7,8,10
3,std004,GP,F,15,U,GT3,T,4,2,health,...,3,2,2,1,1,5,2,15,14,15
4,std005,GP,F,16,U,GT3,T,3,3,other,...,4,3,2,1,2,5,4,6,10,10


### All Grades Table

In [37]:
all_G_df = df_full_math[["student_id", "G1", "G2", "G3"]]

In [38]:
all_G_df.head()

,student_id,G1,G2,G3
0,std001,5,6,6
1,std002,5,5,6
2,std003,7,8,10
3,std004,15,14,15
4,std005,6,10,10


### Zero Count

In [39]:
G1_zero_count = all_G_df[all_G_df['G1'] == 0].shape[0]
G1_zero_count

0

In [40]:
G2_zero_count = all_G_df[all_G_df['G2'] == 0].shape[0]
G2_zero_count

13

In [41]:
all_G_df[all_G_df['G2'] == 0][['G1', 'G2', 'G3']]

,G1,G2,G3
130,12,0,0
131,8,0,0
134,9,0,0
135,11,0,0
136,10,0,0
137,4,0,0
144,5,0,0
153,5,0,0
162,7,0,0
242,6,0,0


* No one scored 0 in G1
* Students who scored 0 in G2 also scored 0 in G3

### Step 1: Calculate the common difference d per student

In [42]:
all_G_df['d'] = all_G_df['G2'] - all_G_df['G1']

In [43]:
all_G_df['d']

0      1
1      0
2      1
3     -1
4      4
      ..
390    0
391    2
392   -2
393    1
394    1
Name: d, Length: 395, dtype: int64

### Step 2 — Calculate expected G3 per student

In [44]:
all_G_df['expected_G3'] = all_G_df['G2'] + all_G_df['d']

In [45]:
all_G_df['expected_G3'] = all_G_df['expected_G3'].clip(upper=20, lower=0)

In [46]:
all_G_df['expected_G3']

0       7
1       5
2       9
3      13
4      14
       ..
390     9
391    18
392     6
393    13
394    10
Name: expected_G3, Length: 395, dtype: int64

In [ ]:
# highest_G = all_G_df['expected_G3'] == 20

In [49]:
# highest_G.value_counts()

expected_G3
False    390
True       5
Name: count, dtype: int64

In [47]:
all_G_df['expected_G3'] = all_G_df['expected_G3'].clip(upper=20, lower=0)

### Step 3 — Calculate deviation per student

In [50]:
deviation = all_G_df['G3'] - all_G_df['expected_G3']

In [51]:
deviation.value_counts()

 0     93
 1     82
-1     58
-2     43
 2     41
 3     21
-3     18
-4     13
-8      7
 5      4
-5      3
 4      3
-11     2
-6      2
-7      2
-9      2
-10     1
Name: count, dtype: int64

### Summary of Findings: The "Momentum" Report

#### 1. The Stability of Arithmetic Progression
* **Finding:** Approximately **55% of the class** follows the arithmetic model very closely (deviation of $-1, 0, \text{or } +1$).
* **Insight:** This proves that G1 and G2 are strong predictors of the final result. Most students don't "change their stripes" mid-semester; they stay on their established path.

#### 2. The "Sudden Detachment" (The G2 Wall)
* **Finding:** No student scored a **0 in G1**, but a significant group dropped to **0 in G2** and stayed there for **G3**.
* **Insight:** Failure in this dataset isn't always a slow decline. For the "Zero-Wall" group, the crisis happened specifically between Period 1 and Period 2. This is the "Critical Zone" for intervention.

#### 3. The Ceiling Effect
* **Finding:** Identified **5 elite students** whose trajectory was so strong they were mathematically "destined" to exceed a perfect score.
* **Insight:** For high achievers, the challenge isn't improvement—it's **maintenance**. Once a student hits 19 or 20, the arithmetic model "breaks" because there is no more room to grow.

#### 4. The "Surprise" Outliers
* **Finding:** The deviation list showed extreme swings, including a **-11 crash** and a **+12 miracle recovery**.
* **Insight:** These students represent the "Human Element." While 55% of the class is predictable, these outliers prove that academic outcomes aren't just math—life events, health, or sudden bursts of motivation can completely flip a student's trajectory regardless of their G1/G2 scores.
